# HW3: Atomic cashflows and the Visitor pattern

This assignment develops two skills:

1. **Model cashflow contracts.** Implement `ProductFixedAccruedCashflow` and `ProductOvernightIndexCashflow` using the provided `Product` and `ProductCashflow` bases. Understand their contractual fields, including dates, notional, currency, accrual conventions, index and spread. Each object is a building block for a larger product, such as a swap.
2. **Use the Visitor pattern.** Collect and display each product's attributes through a visitor. This demonstrates how to add an operation without changing the product classes.

Read [IV. Visitor Pattern](https://immense-chocolate-92e.notion.site/IV-Visitor-Pattern-2d4831d0b1d58191bc36ebae3d771d58); implement `accept()` and the display visitor to produce the tables below.

Complete the six `#TODO` blocks in `fixedincomelib/product/linear_products.py` and `fixedincomelib/product/product_display_visitor.py` before running the examples.

Use the provided `qfCreateProductxxx` APIs to create product objects and `qfDisplayProduct` to inspect their fields.

These objects also form a foundation for a later `value_engine`, which can combine their contract attributes with the necessary rates and market data to calculate cash amounts and present values. The engine reads the product objects directly; the display tables are for inspection.

In [2]:
import sys
import QuantLib as ql

from fixedincomelib import (
    qfCreateProductFixedAccruedCashflow,
    qfCreateProductOvernightIndexCashflow,
    qfDisplayProduct,
)
from fixedincomelib.product import (
    ProductCashflow,
    ProductFixedAccruedCashflow, ProductOvernightIndexCashflow,
)

print(f"Python {sys.version.split()[0]} | QuantLib {ql.__version__}")

Python 3.11.16 | QuantLib 1.41


## 1. Fixed-accrual cashflow

The API returns a `ProductFixedAccruedCashflow` object containing contract fields and an accrual year fraction. `qfDisplayProduct` displays its fields; the fraction is checked below. This class has no fixed-rate field.

In [3]:
fixed = qfCreateProductFixedAccruedCashflow(
    effective_date="2026-01-02",
    termination_date="2026-04-02",
    currency="USD",
    notional=1_000_000,
    accrual_basis="ACT/360",
    payment_date="2026-04-06",
    business_day_convention="F",
    holiday_convention="USGS",
)
assert isinstance(fixed, ProductCashflow)
assert isinstance(fixed, ProductFixedAccruedCashflow)
print("Created:", type(fixed).__name__)
qfDisplayProduct(fixed)

Created: ProductFixedAccruedCashflow


,Name,Value
0,Product Type,PRODUCT_FIXED_ACCRUED
1,Notional,1000000
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-01-02
5,Termination Date,2026-04-02
6,Accrual Basis,ACT/360
7,Payment Date,2026-04-06
8,Business Day Convention,F
9,Holiday Convention,USGS


In [4]:
# 90 calendar days / 360. This number is a year fraction, not a cash amount.
assert fixed.accrued == 90 / 360
assert fixed.termination_date.ISO() == "2026-04-02"
assert fixed.payment_date.ISO() == "2026-04-06"
assert fixed.first_date == fixed.effective_date
assert fixed.last_date == fixed.payment_date
print(f"Accrual year fraction: {fixed.accrued:.6f}")
print("Contractual accrual end:", fixed.termination_date.ISO())
print("Payment date:", fixed.payment_date.ISO())

Accrual year fraction: 0.250000
Contractual accrual end: 2026-04-02
Payment date: 2026-04-06


## 2. Overnight-index cashflow

The API returns a `ProductOvernightIndexCashflow` object. It resolves the tenor to a termination date, obtains currency from the index, and stores the contract fields.

`0.001` is 10 basis points as an annual decimal rate. Negative notional implies `SHORT`. Creation and display do not calculate an interest amount.

In [5]:
overnight = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="3M",
    overnight_index="SOFR-1B",
    notional=-1_000_000,
    compounding_method="compound",
    spread=0.001,
    payment_date="2026-04-06",
)
assert isinstance(overnight, ProductCashflow)
assert isinstance(overnight, ProductOvernightIndexCashflow)
assert overnight.notional == -1_000_000
assert overnight.currency.value_str == "USD"
print("Created:", type(overnight).__name__)
qfDisplayProduct(overnight)

Created: ProductOvernightIndexCashflow


,Name,Value
0,Product Type,PRODUCT_OVERNIGHT_INDEX_CASHFLOW
1,Notional,-1000000
2,Currency,USD
3,Long Or Short,SHORT
4,Effective Date,2026-01-02
5,Termination Date,2026-04-02
6,ON Index,SOFR-1B
7,Compounding Method,COMPOUND
8,Spread,0.001
9,Payment Date,2026-04-06


In [6]:
# Date and tenor inputs resolve to the same termination date.
overnight_by_date = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="2026-04-02",
    overnight_index="SOFR-1B",
    notional=-1_000_000,
    compounding_method="compound",
    spread=0.001,
    payment_date="2026-04-06",
)
assert overnight.termination_date == overnight_by_date.termination_date
assert overnight.payment_date == overnight_by_date.payment_date
print("Date and tenor inputs agree:", overnight.termination_date.ISO())

Date and tenor inputs agree: 2026-04-02


## 3. Currency and payment defaults

Supported indices are `SOFR-1B`, `FF-1B` and `SONIA-1B` (case-insensitive). Currency comes from the index, and an omitted payment date defaults to termination. Compounding labels are `simple`, `arithmetic` and `compound`.

In [7]:
sonia = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="3M",
    overnight_index="SONIA-1B",
    notional=500_000,
    compounding_method="arithmetic",
)
assert sonia.currency.value_str == "GBP"
assert sonia.payment_date == sonia.termination_date
qfDisplayProduct(sonia)

,Name,Value
0,Product Type,PRODUCT_OVERNIGHT_INDEX_CASHFLOW
1,Notional,500000
2,Currency,GBP
3,Long Or Short,LONG
4,Effective Date,2026-01-02
5,Termination Date,2026-04-02
6,ON Index,SONIA-1B
7,Compounding Method,ARITHMETIC
8,Spread,0.0
9,Payment Date,2026-04-02


# Written Part

## 1. Normalized forward price

Setup: 4% semi-annual coupon, exactly 9 years left. "Exactly" means today is a coupon date, so accrued interest is zero, clean = dirty = 100, and no coupon is paid in the next 3 months.

**1) 3M forward price.** The forward is the price at which a cash-and-carry breaks even: buy the bond at 100 today, finance it in repo at 2% for 3 months, deliver at $T_d$. The full (dirty) price I need to receive at delivery is

$$BF^{dirty} = S\,(1 + r_{mkt}\,\tau) = 100\,(1 + 0.02 \times 0.25) = 100.50 .$$

By $T_d$ the bond carries 3 months of accrued interest, $AI(T_d) = 2 \times \tfrac{3}{6} = 1.00$, so the clean forward price is

$$BF = 100.50 - 1.00 = 99.50 .$$

(Clean, because that is what the futures invoice is compared against — see 4.) The forward sits below spot because coupon accrual (1.00) exceeds the financing cost (0.50): positive carry of 0.50.

**2) Conversion factor.** Clean price at a flat 6% yield, 18 semi-annual periods, divided by 100:

$$K = \frac{1}{100}\left[\sum_{k=1}^{18} \frac{2}{1.03^k} + \frac{100}{1.03^{18}}\right] = 0.862465 .$$

(CME would round this to 0.8625; I keep the unrounded value and note the difference where it matters.)

**3) Normalized forward.** $BF/K = 99.50 / 0.862465 = 115.3670$.

**4) What the conversion factor does.** It re-expresses every bond in the basket as the price it would have if it yielded 6%, so a 4% bond ($K<1$) and a 7% bond ($K>1$) are put on the same footing and delivering one or the other is *roughly* a wash. Two things the accrued-interest hint is pointing at:

- $K$ is defined on the clean price, and the invoice the short receives is $F \cdot K + AI(T_d)$: accrued interest is paid on top, not scaled by $K$. So the quantity that belongs over $K$ is the *clean* forward. Dividing the dirty forward would give $100.50/0.862465 = 116.53$, which counts the 1.00 of accrued twice.
- The normalization is only exact if every bond actually yields 6%. Away from 6%, the different durations in the basket make some bonds cheaper to deliver than others — that is the whole reason a CTD exists (Q2–Q3).

## 2. Cheapest-to-deliver

**1) Implied repo.** Set what the future pays at delivery equal to the cost of carrying the bond there, and solve for the financing rate:

$$F K + AI(T_d) = S\,(1 + r^{imp}\tau)
\quad\Rightarrow\quad
r^{imp} = \frac{1}{\tau}\left(\frac{F K + AI(T_d)}{S} - 1\right)
= \frac{1}{0.25}\left(\frac{115.3670 \times 0.862465 + 1.00}{100} - 1\right) = 2.00\% .$$

(With $K$ rounded to 0.8625 this comes out to 2.016%.) Implied repo equals market repo here, which is not a coincidence: the quoted $F$ is exactly the normalized forward from Q1, i.e. the market is pricing this bond as the CTD.

**2) Picking the CTD.** Compute $r^{imp}_x$ for every bond in the basket and take the one with the **highest implied repo**. The reason is the arbitrage in the hint. The short has to go out and buy a bond to deliver. If some bond had $r^{imp}_x > r_{mkt}$, anyone could buy $x$, finance it in GC repo at $r_{mkt}$, sell the future and deliver $x$, pocketing $r^{imp}_x - r_{mkt}$ with no risk. Doing this pushes $F$ down until the best bond has $r^{imp} \approx r_{mkt}$, and every other bond has $r^{imp}_x < r_{mkt}$ — delivering one of those means earning less than it costs to finance it. So "highest implied repo" and "closest to GC from below" pick the same bond, the one attaining the min in Eq. (1).

## 3. Scenario analysis

To a first approximation, yes. Think of each bond's normalized forward $BF_x/K_x$ as a function of the yield level. By construction they all sit near each other at 6%, but with different slopes: a long-maturity, low-coupon bond has more duration than a short-maturity, high-coupon one. So:

- If yields are expected to be **above 6%** in N days, high-duration bonds will have fallen the most relative to their conversion factors — the CTD moves to the **longest-duration** bond in the basket.
- If yields will be **below 6%**, the reverse: the **shortest-duration** (high coupon, short maturity) bond becomes CTD.
- Near 6% the normalized forwards are bunched and the CTD flips easily. A change in curve slope, or one bond going special in repo (cheaper financing lowers its $BF_x$), can also switch it.

Mechanically: take the N-day scenario for yields / curve shape / repo, reprice each bond (`BondCalculator`), recompute $BF_x/K_x$ as in Q1, and the smallest is the predicted CTD.

Why it matters: the future behaves like a forward on the CTD, so both its price and its risk come from that bond — $F \approx BF_{CTD}/K_{CTD}$ and $DV01_{fut} \approx DV01_{CTD}/K_{CTD}$. If the CTD switches, the contract's duration jumps and any hedge built on the old CTD is wrong. The possibility of a switch is also an option held by the short (which bond, and when, to deliver); its value is why the future trades a little below the current CTD's forward (the net basis), and anticipating CTD changes is how that option and the basis trades around it get valued. And if you are long and stand for delivery, you receive whichever bond is cheapest for the short, never the one you would have chosen.

The cell below runs this for a small basket of hypothetical bonds under a few flat-yield scenarios, and also checks the numbers in Q1 and Q2.

In [8]:
import numpy as np
from fixedincomelib.analytics.bond_calculator import BondCalculator

today, repo, tau = '2026-10-15', 0.02, 0.25   # today is a coupon date, delivery in 3M


def make_bond(coupon, years):
    # bullet bond issued one year ago on a coupon date, so accrued today is zero
    y0 = int(today[:4])
    conv = {
        'issue date': f'{y0-1}-10-15', 'first accrual date': f'{y0-1}-10-15',
        'first coupon date': f'{y0}-04-15',
        'maturity date': f'{y0+years}-10-15', 'last regular coupon date': f'{y0+years}-04-15',
        'coupon rate': coupon, 'face value': 100.0, 'redemption': 1.0,
        'coupon accrual period': '6M', 'schedule rule': 'BACKWARD', 'end of month': False,
        'accrual basis': 'ACT/ACT (ICMA)', 'first period accrual basis': 'ACT/ACT (ICMA)',
        'last period accrual basis': 'ACT/ACT (ICMA)',
        'settlement offset': '0D', 'settlement business day convention': 'F',
        'settlement holiday convention': 'USGS',
        'payment offset': '0D', 'payment business day convention': 'F',
        'payment holiday convention': 'USGS',
    }
    return BondCalculator(bond_convention=conv)


def normalized_forward(bond, spot_dirty):
    ai_Td = 100 * bond.coupon_rate * tau                    # accrued at delivery, no coupon in between
    bf_clean = spot_dirty * (1 + repo * tau) - ai_Td
    K = bond.yield_to_price(settlement_date=today, yield_rate=0.06)['clean_price'] / 100
    return bf_clean, K, ai_Td


# Q1: 4% semi-annual, exactly 9Y left, spot 100
bond = make_bond(0.04, 9)
S = 100.0
bf, K, ai_Td = normalized_forward(bond, S)
print(f"Q1  BF dirty = {S*(1+repo*tau):.4f}   AI(Td) = {ai_Td:.4f}   BF clean = {bf:.4f}")
print(f"    K = {K:.6f}   BF/K = {bf/K:.4f}")

# Q2: implied repo from F = 115.3670
F = 115.3670
r_imp = ((F * K + ai_Td) / S - 1) / tau
r_imp_4dp = ((F * round(K, 4) + ai_Td) / S - 1) / tau
print(f"Q2  implied repo = {r_imp:.2%}   (K rounded to 0.8625: {r_imp_4dp:.3%})")

# Q3: who is CTD in a small basket when the whole curve sits at 4% / 6% / 8%
basket = {'2.5% 7Y': make_bond(0.025, 7), '4% 9Y': make_bond(0.04, 9), '5% 10Y': make_bond(0.05, 10)}
for y in (0.04, 0.06, 0.08):
    row = {}
    for name, b in basket.items():
        spot = b.yield_to_price(settlement_date=today, yield_rate=y)['dirty_price']
        bf, K, _ = normalized_forward(b, spot)
        row[name] = bf / K
    ctd = min(row, key=row.get)
    print(f"Q3  yield {y:.0%}: " + "   ".join(f"{k} -> {v:.3f}" for k, v in row.items()) + f"   CTD = {ctd}")

Q1  BF dirty = 100.5000   AI(Td) = 1.0000   BF clean = 99.5000
    K = 0.862465   BF/K = 115.3670
Q2  implied repo = 2.00%   (K rounded to 0.8625: 2.016%)
Q3  yield 4%: 2.5% 7Y -> 113.110   4% 9Y -> 115.367   5% 10Y -> 116.103   CTD = 2.5% 7Y
Q3  yield 6%: 2.5% 7Y -> 99.721   4% 9Y -> 99.341   5% 10Y -> 99.150   CTD = 5% 10Y
Q3  yield 8%: 2.5% 7Y -> 88.096   4% 9Y -> 85.864   5% 10Y -> 85.092   CTD = 5% 10Y
